# Probe Guidance: World Modeling with MPC Example

In [1]:
import sys
sys.path.insert(0, "/home/jack/code/vjepa2-probe-guidance/vjepa2")
print(sys.path)

['/home/jack/code/vjepa2-probe-guidance/vjepa2', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2-probe-guidance/vjepa2/src', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/rerun_sdk']


In [2]:
import copy
import os

import numpy as np
import torch

from app.vjepa_ll_probe_guidance.ll_probe_guidance import LLProbeGuidanceDataset
from app.vjepa_ll_probe_guidance.utils import init_video_model
from app.vjepa_ll_probe_guidance.transforms import make_transforms
from notebooks.ultrasound_probe_guidance.world_model_wrapper import WorldModel

/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Model Predictive Control (MPC)
VJEPA2 uses MPC to perform tasks like controlling a robot arm to accomplish some goal. The "predictor" that we trained is used to understand how actions affect the state of the world. Then, the cross entropy method (CEM) is used to do "optimization", i.e. it will find a set of actions that minimize error/costs. The error in this case would be, "does the VJEPA2 AC predictor think that this action will reduce the L1 distance to the goal state"?. Then it does "receding horizon" control by simply taking the first action in the action trajectory that CEM found.

### MPC Example

In [3]:
def load_state_dict_with_ddp_fix(model, state_dict):
    new_state_dict = {}
    for k, v in state_dict.items():
        # Remove 'module.' prefix if it exists
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict, strict=True)
    return model

def load_clips(sample, device):
    clips = sample[0].to(device, non_blocking=True)  # [B C T H W]
    actions = sample[1].to(device, non_blocking=True)  # [B T-1 6]
    states = sample[2].to(device, non_blocking=True)  # [B T 6]
    extrinsics = sample[3].to(device, non_blocking=True)  # [B T 6]
    return (clips, actions, states, extrinsics)

In [4]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

encoder, predictor = init_video_model(
        device=device,
        patch_size=16,
        max_num_frames=512,
        tubelet_size=2,
        model_name="vit_large",
        crop_size=256,
        pred_depth=12,
        pred_num_heads=12,
        pred_embed_dim=768,
        action_embed_dim=6,
        predictor_type="ac",
        pred_is_frame_causal=True,
        use_extrinsics=False,
        use_sdpa=True,
        use_rope=True
    )
target_encoder = copy.deepcopy(encoder)

encoder.eval()
predictor.eval()
target_encoder.eval()

resume_path = os.path.join("/home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4", "best.pt")
if os.path.exists(resume_path):
    print(f"Loading checkpoint from {resume_path}")
    checkpoint = torch.load(resume_path, map_location=torch.device("cpu"))
    encoder = load_state_dict_with_ddp_fix(encoder, checkpoint["encoder"])
    predictor = load_state_dict_with_ddp_fix(predictor, checkpoint["predictor"])
    target_encoder = load_state_dict_with_ddp_fix(target_encoder, checkpoint["target_encoder"])
else:
    print(f"Checkpoint not found at {resume_path}")

print("=" * 20 + "PREDICTOR" + "=" * 20)
print(predictor)
print("=" * 20 + "TARGET ENCODER" + "=" * 20)
print(target_encoder)

Loading checkpoint from /home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4/best.pt
====================PREDICTOR====================
VisionTransformerPredictorAC(
  (predictor_embed): Linear(in_features=1024, out_features=768, bias=True)
  (action_encoder): Linear(in_features=6, out_features=768, bias=True)
  (state_encoder): Linear(in_features=6, out_features=768, bias=True)
  (extrinsics_encoder): Linear(in_features=5, out_features=768, bias=True)
  (predictor_blocks): ModuleList(
    (0-11): 12 x ACBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): ACRoPEAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, b

In [5]:
crop_size = 256
tokens_per_frame = int((crop_size // encoder.patch_size) ** 2)
transform = make_transforms(
    crop_size=crop_size,
)

# SET THIS TO THE NUMBER OF FRAMES YOU WANT IN A CLIP
T = 8

val_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/test",
    frames_per_clip=T,
    frame_skip=1,
    frames_per_second=4,
    transform=transform,
    is_train=False,
)

loader = torch.utils.data.DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=1, # make sure batch size is always 1 for CEM
    drop_last=True,
    pin_memory=True,
    num_workers=8,
)

Scanning 7 episodes for valid tracking clips...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 38.16it/s]

Retained 7 episodes.


In [6]:
world_model = WorldModel(
    encoder=encoder,
    predictor=predictor,
    tokens_per_frame=tokens_per_frame,
    mpc_args={
        "rollout": 1,
        "samples": 50,
        "topk": 10,
        "cem_steps": 10,
        "momentum_mean": 0.15,
        "momentum_std": 0.75,
        "maxnorm": 0.075,
        "verbose": False
    },
    normalize_reps=True,
    device=device
)

sample = next(iter(loader))
clips, actions, states, _ = load_clips(sample, device)
#print(f"{clips.shape=}, {actions.shape=}, {states.shape=}")

with torch.no_grad():
    h = world_model.encode(clips)
    print(f"{h.shape=}")
    z_n, z_goal = h[:, :tokens_per_frame], h[:, tokens_per_frame:tokens_per_frame*2]
    s_n = states[:, :1]
    print(f"Starting planning using Cross-Entropy Method...")
    wm_actions = world_model.infer_next_action(z_n, s_n, z_goal).cpu().numpy()

print(f"First action returned by planning with CEM (x,y,z) = ({wm_actions[0, 0]:.5f}, {wm_actions[0, 1]:.5f}, {wm_actions[0, 2]:.5f})")
print(f"Ground truth first action (x,y,z) = ({actions[0, 0, 0]:.5f}, {actions[0, 0, 1]:.5f}, {actions[0, 0, 2]:.5f})")

/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


h.shape=torch.Size([1, 2048, 1024])
Starting planning using Cross-Entropy Method...
First action returned by planning with CEM (x,y,z) = (0.06846, -0.00614, -0.05023)
Ground truth first action (x,y,z) = (0.01366, 0.00129, -0.00253)
